# 01 — Data Cleaning (walkthrough)

**Why (lesson 1.7 Pandas):** We clean in a script (`01_clean_data.py`) for reproducibility,
but walk through the logic here so you can see *each rule* against the live data.

Each cell below = one group cleaning rule.

In [ ]:
import duckdb, pandas as pd, numpy as np
from pathlib import Path
DB = Path('../data/sg_jobs.duckdb')
con = duckdb.connect(str(DB), read_only=True)
raw = con.execute('SELECT * FROM raw_jobs').df()
clean = con.execute('SELECT * FROM clean_jobs').df()
print('raw rows   :', len(raw))
print('clean rows :', len(clean))
raw[['average_salary','numberOfVacancies','minimumYearsExperience']].describe()

## Rule 15 — LQS drop ($1,800)

**Why:** The LQS sets the minimum monthly wage for full-time locals ($1,800).
We follow the group decision to drop jobs below this, **but** we keep the count so the
dashboard can cite how many jobs were excluded. Part-time jobs are assessed hourly under LQS,
so they are kept in demand metrics but excluded from the monthly LQS threshold chart.

In [ ]:
print('clean avg_salary describe:')
clean['average_salary'].describe()

## Rule 11 — PMET mapping

**Why:** EP qualifying salary benchmarks are PMET-only. We need `is_pmet`
to compute PMET wage percentiles.

In [ ]:
clean['is_pmet'].value_counts()

## Rule 12 — normalised job_categories

**Why (lesson 1.5 advanced SQL):** `categories` is a JSON array of objects;
`unnest` + `json_extract_string` gives one row per (job, category), which we can
join to `clean_jobs` on `metadata_jobPostId = job_post_id`.

In [ ]:
con.execute('SELECT * FROM job_categories LIMIT 5').df()